# GLARE Methodology: pipeline runner (Assignment 2, Group 1)

**Status: skeleton.** This notebook only runs the modules in the repository, one stage per section, and shows the result of each stage. The logic lives in `glare_issues/`. Each section is owned by the member named in its heading, who edits their own cell through a pull request.

**Rules**
- Every number reported in the final document must come from a cell of this notebook, never from the demo mode.
- Same seed (42) everywhere. LLM calls at temperature 0, answers cached on disk.
- Save the output of every stage to Google Drive (last cell of each section), so a disconnected Colab can continue from the last checkpoint.
- Code comes from the public GitHub repository (default, `CODE_SOURCE = 'github'`) or from `code.zip` on the shared Drive folder (set `CODE_SOURCE = 'drive'`).
- Use a GPU for the embedding stage: Runtime, Change runtime type, T4 GPU.

**Open decisions (fill in, then delete this line):** LLM name and version = `TODO`, Model C checkpoint = `TODO`, sample size = `30,000`.

## 0. Setup (everyone)

In [1]:
# Settings: change only after agreeing in the group chat
DEMO = False               # True = synthetic data, mock LLM, offline embedder (smoke test, NOT results). Set False for real runs.
CODE_SOURCE = 'github'    # 'drive' = unzip code.zip from the shared Drive folder (simple, no token); 'github' = clone the repo (public: no token, private: token in Colab Secrets)
REPO_URL = 'https://github.com/layan-sa1/glare-issue-discovery.git'   # only used when CODE_SOURCE = 'github'
BRANCH = 'main'           # to test your own changes before merging, write your branch name here (for example 'member3-model-c')
DRIVE_DIR = '/content/drive/MyDrive/GLARE_Project'                     # where checkpoints are saved
CODE_ZIP = DRIVE_DIR + '/code.zip'                                       # GitHub, Code, Download ZIP, then upload it here
MASTER_POOL_ZIP = DRIVE_DIR + '/GLARE_Master_Pool_1M.zip'               # same file used in the pilot
TARGET_SAMPLE = 30_000    # TODO: agree on the final sample size
LLM_BACKEND = 'gemini'    # 'anthropic' or 'openai' or 'gemini' (ignored when DEMO = True)
LLM_MODEL = 'gemini-3.1-flash-lite'            # TODO: exact model name and version, also write it in the report
SEED = 42

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

Mounted at /content/drive


In [6]:
# Get the code
import os, shutil, glob
%cd /content
shutil.rmtree('repo', ignore_errors=True)   # safe to rerun this cell
if CODE_SOURCE == 'drive':
    # Update workflow: on GitHub press Code, Download ZIP, upload the file to DRIVE_DIR as code.zip (replace the old one)
    shutil.rmtree('repo', ignore_errors=True)
    !unzip -q "{CODE_ZIP}" -d repo_tmp
    inner = [p for p in glob.glob('repo_tmp/*') if os.path.isdir(p)]
    shutil.move(inner[0] if len(inner) == 1 else 'repo_tmp', 'repo')
    shutil.rmtree('repo_tmp', ignore_errors=True)
else:
    # Public repository: no token needed. Private repository: save a GitHub token in Colab Secrets (key icon) under the name GH_TOKEN.
    url = REPO_URL
    try:
        from google.colab import userdata
        url = REPO_URL.replace('https://', f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        print('No GH_TOKEN found, cloning without a token (works only for a public repository).')
    !git clone -q -b {BRANCH} {url} repo
%cd repo
!pip install -q -r requirements.txt

/content
No GH_TOKEN found, cloning without a token (works only for a public repository).
/content/repo


In [ ]:
# Code changed on GitHub? Drive mode: upload a fresh code.zip and rerun the cell above. GitHub mode: run !git pull here.

In [11]:
import sys, importlib
sys.path.insert(0, '.')
import pandas as pd, numpy as np
import run_pipeline as R
from glare_issues.config import Config
from glare_issues.llm import make_llm

cfg = Config(out_dir='results', llm_backend=LLM_BACKEND, llm_model=LLM_MODEL, target_sample=TARGET_SAMPLE)
cfg.seed = SEED
embedders = cfg.embedders
if DEMO:
    from glare_issues.demo_data import make_demo_reviews
    cfg.out_dir, cfg.cache_dir, cfg.llm_backend = 'demo_results', 'demo_cache', 'mock'
    cfg.n_trials, cfg.kmeans_k_range = 8, [5, 7, 10, 15]
    cfg.umap_space = {'n_neighbors': [10, 15], 'n_components': [5, 10], 'min_dist': [0.0, 0.1]}
    cfg.hdbscan_space = {'min_cluster_size': [10, 20, 40], 'min_samples': [1, 5], 'cluster_selection_method': ['eom', 'leaf']}
    embedders = {'DemoTFIDF-A': {'kind': 'tfidf', 'name': ''}}
    print('DEMO MODE: these numbers are NOT research results.')
llm = make_llm(cfg)
print('Output folder:', cfg.out_dir)

Output folder: results


## 1. Data and preprocessing (Member 1)
Reads the Master Pool, cleans, merges duplicates (keeping counts), applies the per-app cap. Output: `01_preprocessed.csv`.

Done (Member 1): reads the Master Pool, keeps reviews with at least two words, and writes the sample to `01_preprocessed.csv`. Sample size and number of apps are printed below.

In [ ]:
import pandas as pd

if DEMO:
    df = make_demo_reviews()
else:
    df = pd.read_csv(
        MASTER_POOL_ZIP,
        usecols=['review_id', 'app_id', 'content', 'rating', 'at'],
        dtype={'review_id': 'string', 'app_id': 'string', 'content': 'string'},
    )
    print('Rows read:', len(df))
    print('Apps:', df['app_id'].nunique())
    cfg.min_words = 2   # drop one-word reviews (same rule as the pilot)

d = R.stage_prepare(cfg, df)
display(d.head())

per_app = d.groupby('app_id').size()
print('Apps in sample:', per_app.size)
print(per_app.describe())
for k in (1, 2):
    n = (d['n_words'] == k).sum()
    print(f'{k}-word texts:', n, f'({n / len(d):.1%})')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Data exploration figure (Member 1): why we filter and cap the sample
os.makedirs(cfg.out_dir, exist_ok=True)
raw_words = df['content'].astype('string').fillna('').str.split().str.len()
raw_per_app = df.groupby('app_id').size()
sample_per_app = d.groupby('app_id').size()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# (a) review length in the full pool
ax[0].hist(raw_words.clip(upper=40), bins=np.arange(0.5, 41.5, 1), color='#4C72B0')
ax[0].axvline(cfg.min_words - 0.5, color='crimson', ls='--', label=f'min_words = {cfg.min_words}')
ax[0].set_xlabel('Words per review (40 = 40 or more)'); ax[0].set_ylabel('Reviews')
ax[0].set_title('(a) Review length, full pool'); ax[0].legend()

# (b) rating distribution, pool versus sample
pool_r = df['rating'].round().value_counts(normalize=True).sort_index()
samp_r = d['rating_mean'].round().value_counts(normalize=True).sort_index()
x = np.arange(1, 6); w = 0.38
ax[1].bar(x - w/2, [pool_r.get(i, 0) for i in x], w, label='Full pool', color='#4C72B0')
ax[1].bar(x + w/2, [samp_r.get(i, 0) for i in x], w, label='Sample', color='#DD8452')
ax[1].set_xlabel('Rating'); ax[1].set_ylabel('Share of reviews')
ax[1].set_title('(b) Rating distribution'); ax[1].legend()

# (c) reviews per app, pool versus sample
bins = np.logspace(0, np.log10(max(raw_per_app.max(), 10)), 30)
ax[2].hist(raw_per_app, bins=bins, alpha=0.7, label='Full pool', color='#4C72B0')
ax[2].hist(sample_per_app, bins=bins, alpha=0.7, label='Sample (after cap)', color='#DD8452')
ax[2].set_xscale('log'); ax[2].set_yscale('log')
ax[2].set_xlabel('Reviews per app (log scale)'); ax[2].set_ylabel('Apps (log scale)')
ax[2].set_title('(c) Reviews per app'); ax[2].legend()

plt.tight_layout()
path = os.path.join(cfg.out_dir, '01_data_exploration.png')
fig.savefig(path, dpi=200)
plt.show()

print(f'Full pool: {len(df):,} reviews, {raw_per_app.size:,} apps, median {raw_per_app.median():.0f} reviews per app, max {raw_per_app.max():,}')
print(f'Sample: {len(d):,} unique texts, {sample_per_app.size:,} apps, median {sample_per_app.median():.0f} per app, max {sample_per_app.max()}')
print(f'Reviews under {cfg.min_words} words in the pool: {(raw_words < cfg.min_words).mean():.1%}')
print('Figure saved to', path)

In [ ]:
# Checkpoint to Drive
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 2. Stage 1: problem filter (Member 5)
LLM classifies each unique review as problem report or not. Outputs: `02_filtered_all.csv`, `02_problem_reviews.csv`.

TODO (Member 5): choose the LLM, run a 200-review pilot against human labels, record the model name and version.

In [ ]:
# Stage 1: Problem Filter - Member 5

import pandas as pd
from pathlib import Path

results_dir = Path(DRIVE_DIR) / "results"

all_file = results_dir / "02_filtered_all.csv"
problem_file = results_dir / "02_problem_reviews.csv"

if all_file.is_file() and problem_file.is_file():
    print("Saved results found. Skipping classification.")

    d_all = pd.read_csv(all_file)
    kept = pd.read_csv(problem_file)

else:
    print("Saved results not found. Running classification.")

    d_all, kept = R.stage_filter(cfg, d, llm)

print("Total reviews:", len(d_all))
print("Problem reviews:", len(kept))

display(kept.head())


In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 3. Annotation sheets (Member 6)
Writes blank sheets for the human parts of the evaluation (filter labels, issue grouping). Annotators fill the last column and save the file with the name plus `_done`, in `results/annotations/`.

In [ ]:
R.stage_annotate(cfg, d_all, kept)

In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 4. Embeddings, UMAP, HDBSCAN, K-Means (Members 2, 3, 4)
Members 2 and 3 own the embedding models A, B and C (`glare_issues/embed.py`, settings in `config.py`). Member 4 owns UMAP, HDBSCAN, K-Means and the random search (`glare_issues/cluster.py`). Needs a GPU for real runs.

TODO (Member 3): replace the placeholder Model C checkpoint in `config.py`.

In [ ]:
# Member 4: keep embeddings and clustering results on Drive, and reload the reviews after a disconnect
if not DEMO:
    cfg.cache_dir = DRIVE_DIR + '/cache'
    os.makedirs(cfg.cache_dir, exist_ok=True)
    if 'kept' not in globals():
        kept = pd.read_csv(DRIVE_DIR + '/results/02_problem_reviews.csv')
        d_all = pd.read_csv(DRIVE_DIR + '/results/02_filtered_all.csv')

In [ ]:
table, store = R.stage_cluster(cfg, kept, embedders)
display(table)

In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 5. Stage 3: issue descriptions and ranking (Member 5)
Representative reviews, LLM label and Arabic summary, ranking by occurrences. Output: `04_ranked_issues.csv`.

In [18]:
# Load the clustering comparison
table = pd.read_csv(
    os.path.join(cfg.out_dir, "03_config_comparison.csv")
)

# Use the clustered reviews
kept = clusters.copy()

# Prepare Model C embeddings and labels
labels = np.array(cluster_cache["lab_h"])

store = {
    ("ArabicSBERT", "HDBSCAN"): (X, labels)
}

# Verify required data
print("Reviews:", len(kept))
print("Embeddings:", X.shape)
print("Clusters:", len(set(labels) - {-1}))
print("Store ready:", ("ArabicSBERT", "HDBSCAN") in store)
print("Required columns:", all(c in kept.columns for c in ["text", "count"]))

Reviews: 6313
Embeddings: (6313, 768)
Clusters: 42
Store ready: True
Required columns: True


### Step 1: Test LLM Summarization on One Cluster

Before summarizing all 42 clusters, we tested Gemini on one cluster (ID 27) to verify that the summarization process works correctly.

In [19]:
# Prepare a safe test configuration
import copy
import os

test_cfg = copy.copy(cfg)

# Test only one cluster
test_cfg.summary_top_clusters = 1

# Save test results separately
test_cfg.out_dir = "/content/glare_summary_test"

os.makedirs(test_cfg.out_dir, exist_ok=True)
os.makedirs(
    os.path.join(test_cfg.out_dir, "annotations"),
    exist_ok=True
)

print("Test clusters:", test_cfg.summary_top_clusters)
print("Test output:", test_cfg.out_dir)
print("Selected model: ArabicSBERT + HDBSCAN")

Test clusters: 1
Test output: /content/glare_summary_test
Selected model: ArabicSBERT + HDBSCAN


In [20]:
res_test = R.stage_summarize(
    test_cfg,
    kept,
    store,
    table,
    llm,
    choice=("ArabicSBERT", "HDBSCAN")
)

print("Summarization completed!")
print(res_test[["cluster", "label", "summary_ar"]].to_string(index=False))

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[summarize] ('ArabicSBERT', 'HDBSCAN'): 1 issue descriptions written (LLM calls: 1)
Summarization completed!
 cluster           label                                                                     summary_ar
      27 App not working يواجه المستخدمون مشاكل في تشغيل اللعبة حيث يصفها البعض بأنها لا تعمل أو معطلة.


In [39]:
import os
import pandas as pd
import numpy as np
from glare_issues import cluster as C

# Prepare all clusters (exclude noise)
ranked = C.rank_clusters(labels, kept["count"].values)
all_clusters = ranked[ranked["cluster"] != -1].copy()

# Select representative reviews and top terms
reps = C.representatives(X, labels, cfg.summary_samples)
terms = C.cluster_top_terms(kept["text"].tolist(), labels)

# Folder for individual cluster summaries
summary_dir = os.path.join(cfg.out_dir, "summary_progress")
os.makedirs(summary_dir, exist_ok=True)

# Summarize all clusters
for i, cluster_id in enumerate(all_clusters["cluster"], start=1):
    cluster_id = int(cluster_id)
    save_path = os.path.join(summary_dir, f"cluster_{cluster_id}.csv")

    # Skip already saved clusters
    if os.path.exists(save_path):
        print(f"[{i}/42] Cluster {cluster_id}: Already saved")
        continue

    row = all_clusters[all_clusters["cluster"] == cluster_id].iloc[0]
    sample = kept.iloc[reps[cluster_id]]["text"].tolist()

    try:
        result = llm.summarise_cluster(
            sample,
            int(row["unique_reviews"])
        )

        # Handle multiple issues returned by Gemini
        if isinstance(result, list):
            if not result or not all(isinstance(x, dict) for x in result):
                raise ValueError("Unexpected Gemini response format")

            result = {
                "label": "Multiple issues" if len(result) > 1 else result[0].get("label"),
                "summary_ar": " | ".join(
                    f"{x.get('label', '')}: {x.get('summary_ar', '')}"
                    for x in result
                )
            }

        output = pd.DataFrame([{
            "rank": i,
            "cluster": cluster_id,
            "unique_reviews": int(row["unique_reviews"]),
            "occurrences": int(row["occurrences"]),
            "label": result.get("label"),
            "summary_ar": result.get("summary_ar"),
            "top_terms": " ".join(terms[cluster_id]),
            "sample_reviews": " || ".join(sample[:3])
        }])

        output.to_csv(
            save_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"[{i}/42] Cluster {cluster_id}: Saved!")

    except Exception as e:
        print(f"Stopped at Cluster {cluster_id}: {e}")
        break

print("Finished current run.")

[1/42] Cluster 27: Already saved
[2/42] Cluster 21: Already saved
[3/42] Cluster 38: Already saved
[4/42] Cluster 20: Already saved
[5/42] Cluster 1: Already saved
[6/42] Cluster 30: Already saved
[7/42] Cluster 36: Already saved
[8/42] Cluster 22: Already saved
[9/42] Cluster 33: Already saved
[10/42] Cluster 17: Already saved
[11/42] Cluster 34: Already saved
[12/42] Cluster 15: Already saved
[13/42] Cluster 24: Already saved
[14/42] Cluster 18: Already saved
[15/42] Cluster 5: Already saved
[16/42] Cluster 6: Already saved
[17/42] Cluster 12: Already saved
[18/42] Cluster 13: Already saved
[19/42] Cluster 7: Already saved
[20/42] Cluster 9: Already saved
[21/42] Cluster 2: Already saved
[22/42] Cluster 29: Already saved
[23/42] Cluster 41: Already saved
[24/42] Cluster 10: Already saved
[25/42] Cluster 40: Already saved
[26/42] Cluster 8: Already saved
[27/42] Cluster 19: Already saved
[28/42] Cluster 32: Already saved
[29/42] Cluster 0: Already saved
[30/42] Cluster 25: Already sav

In [40]:
import os
import pandas as pd

# Collect all 42 saved summaries
all_results = []

for cluster_id in all_clusters["cluster"]:
    path = os.path.join(
        summary_dir,
        f"cluster_{int(cluster_id)}.csv"
    )

    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing Cluster {cluster_id}")

    all_results.append(pd.read_csv(path))

# Combine and sort by rank
final_results = pd.concat(
    all_results, ignore_index=True
).sort_values("rank").reset_index(drop=True)

# Save final results
final_path = os.path.join(
    cfg.out_dir, "04_ranked_issues.csv"
)

final_results.to_csv(
    final_path,
    index=False,
    encoding="utf-8-sig"
)

print("Total summaries:", len(final_results))
print("Saved to:", final_path)
print(final_results[["rank", "cluster", "label"]].to_string(index=False))

Total summaries: 42
Saved to: /content/drive/MyDrive/GLARE_Project/results/04_ranked_issues.csv
 rank  cluster                               label
    1       27                     App not working
    2       21                     App not working
    3       38                       App is a scam
    4       20                     App not working
    5        1            Excessive advertisements
    6       30 App performance and freezing issues
    7       36    Performance and technical issues
    8       22       Login and Registration Issues
    9       33                 App fails to launch
   10       17                         High prices
   11       34                 App fails to launch
   12       15          Negative update experience
   13       24                     App not working
   14       18               Very poor app quality
   15        5 Misleading or missing image content
   16        6     Audio and music playback issues
   17       12     Video saving and q

## 6. Evaluation (Member 6)
Reads the completed annotation sheets (`*_done.csv`). If a sheet is missing, that part is skipped. Output: `05_evaluation_report.json`, `05_external_scores.csv`, figures.

TODO (Member 6): copy the numbers into the result tables of the report (Tables in Section 6.5) and into the benchmarking comparison.

In [ ]:
report = R.stage_evaluate(cfg, kept, d_all)
report

In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 7. Final check (everyone)
- [ ] `DEMO = False`, and the whole notebook runs from the first cell to the last on a fresh Colab session.
- [ ] LLM name and version, Model C checkpoint and sample size are filled in at the top and written in the report.
- [ ] Every table and figure in the report comes from this notebook.
- [ ] No API keys or tokens appear in any cell output.